# Declare → Diagnose → Redesign: The Design Engine

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb11_declare_diagnose_redesign_student.ipynb)

---

## 🧭 Approach & Claim Boundary

**Approach emphasis:** all four approaches, seen through the **diagnosis engine** —
the course's flight simulator. You *declare* a design in code, *diagnose* how it
would behave across a thousand imagined runs, and *redesign* it to fix what the
diagnosis exposes — all before a single real data point is collected.

| | |
|---|---|
| **A claim this topic PERMITS** | "Simulated 1,000 times, my design shows bias ≈ B, power ≈ P, coverage ≈ C for my inquiry — measured, not assumed. Redesign R raises power from P₁ to P₂ at cost C." |
| **A claim this topic does NOT permit** | "My design is fine" as a feeling; or quietly changing the QUESTION to whatever the weak design can answer and calling it a redesign. |

**Where this sits in the course:** meetings M23–M24 of 44 (Mon Oct 19, Wed Oct 21).
It develops milestone M08 — your Design Diagnosis & Redesign Memo. It takes the
mentoring world from nb04 and the by-hand simulations you built all through nb10
and packages them into two reusable tools you will run on your own design.

*Provenance: RDSS ch. 10 'Diagnosing designs' + ch. 11 'Redesigning' + ch. 13 'Designing in code' | declare/diagnose/redesign loop | declaration_10.1 / diagnosis_10.1 / declaration_11.1 concepts translated R→Python into a compact inline helper | translated (R→Python)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Declare** a design as a small function that returns an estimate, its standard
   error, and the truth it is aiming at.
2. Run a **1,000-replication diagnosis** and read the three **diagnosands** —
   **bias**, **power**, and **coverage** — off the result.
3. **Match a sick design to its disease** — bias, low power, or bad coverage — from
   its diagnosand table, and explain what causes each.
4. **Redesign** across a parameter grid, build a comparison table, and pick the
   **cheapest fix** that repairs the weakness.
5. Tell an **honest redesign** (change the design, keep the question) from a
   **silent** one (quietly shrink the question to whatever the design can answer).
6. Declare and diagnose a simplified version of **your own design (M08)**, and
   commit to one redesign with before/after numbers.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

In [ ]:
# The mentoring-program world returns one more time — same as nb04 and nb10.
# Each person has a no-mentor score Y0 and a with-mentor score Y1 = Y0 + effect,
# so the true average effect is EXACTLY `effect`.

def make_world(n=100, effect=2.0, noise=2.0, rng=rng):
    """n people, each with a no-mentor score Y0 and a with-mentor score Y1."""
    Y0 = rng.normal(50, 10, n) + rng.normal(0, noise, n)
    return pd.DataFrame({"Y0": Y0.round(1), "Y1": (Y0 + effect).round(1)})

def complete_ra(n, m, rng):
    """Randomly assign exactly m of n people to the mentor (Z=1), the rest to Z=0."""
    z = np.zeros(n, dtype=int); z[:m] = 1
    return rng.permutation(z)

def difference_in_means(df, y="Y", z="Z"):
    """Treated mean minus control mean, plus the standard error of that difference."""
    g1, g0 = df.loc[df[z] == 1, y], df.loc[df[z] == 0, y]
    est = g1.mean() - g0.mean()
    se = np.sqrt(g1.var(ddof=1) / len(g1) + g0.var(ddof=1) / len(g0))
    return est, se

print("✓ The mentoring world is loaded — Y0, Y1, and a true effect of 2.0.")

## 1. Why This Matters

> *"Pilots don't earn their wings by crashing real planes. They crash a thousand
> times in a simulator, where the only thing that dies is a bad assumption. Your
> research design deserves the same courtesy: crash it in the computer, where the
> cost is a few seconds — not a wasted semester and a room full of people whose
> time you spent collecting data a broken design could never use."*
> — a **design methodologist**, on why you simulate a study before you run it

Everything you did by hand in nb10 — replicate the design, watch the estimates
scatter, measure the spread — was the raw material of a single idea: you can
interrogate a design *before* you run it. This notebook turns that idea into an
**engine** with two moves.

**Declare** a design: write it as a small function that, each time it is called,
plays out one full study and hands back three things — the **estimate** it
produced, the **standard error** it reported, and the **truth** it was aiming at
(which you know, because you built the world). **Diagnose** it: call that function
1,000 times and summarize how it behaved into three numbers, the **diagnosands**:

- **Bias** — on average, how far off the truth does the estimate land? (Should be \~0.)
- **Power** — how often does the design detect a real effect? (Higher is better.)
- **Coverage** — how often does the 95% interval actually contain the truth?
  (Should be \~0.95 — an honest interval keeps its promise.)

> **A question that often comes up here:** *"This sounds like I need to be a
> programmer to design a study."* You don't. The machinery — the loop, the array
> math — is written for you and narrated below; you will never author it. Your job
> is the researcher's job: choose the **parameters** (how many people, how big an
> effect you care about, how noisy your measure is), run the diagnosis, and read
> what it tells you. The engine turns the crank; you decide where to point it.

---

> 💡 **Gemini Prompt:** "This code defines two functions: one plays out a single randomized two-group study and returns its estimate, its standard error, and the true effect; the other runs that study 1,000 times and summarizes three diagnosands — bias, power, and coverage: [paste the next cell]. Explain in plain words exactly what each of the three diagnosands measures, and what value each SHOULD take for an honest, well-powered design."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check Gemini's description of `coverage` against the code: it counts the share of runs whose 95% interval actually contains the truth, so an honest design lands near 0.95 — confirm that is what the formula does.
> - [ ] Confirm `bias` is the average of (estimate − truth), so a design that aims true sits near 0 — not that any single run must be exactly right.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# THE ENGINE (read it, don't write it). Two pieces, exactly as in the API mapping.
#
# 1) A DESIGN is a function that runs ONE study and returns (est, se, truth).
#    You only ever change the PARAMETERS at the top: n, effect, noise.
def two_group_design(rng, n=100, effect=2.0, noise=2.0):
    world = make_world(n, effect, noise, rng)          # declare the world
    world["Z"] = complete_ra(n, n // 2, rng)           # assign half to a mentor
    world["Y"] = np.where(world["Z"] == 1, world["Y1"], world["Y0"])  # reveal
    est, se = difference_in_means(world)               # estimate
    return est, se, effect                             # truth is `effect`, by construction

# 2) DIAGNOSE runs a design `reps` times and summarizes the three diagnosands.
def diagnose(design_fn, reps=1000, seed=464):
    rng = np.random.default_rng(seed)
    rows = [design_fn(rng) for _ in range(reps)]       # 1,000 imagined studies
    d = pd.DataFrame(rows, columns=["est", "se", "truth"])
    return pd.Series({
        "bias":     (d.est - d.truth).mean(),
        "power":    (np.abs(d.est / d.se) > 1.96).mean(),
        "coverage": ((d.est - 1.96 * d.se <= d.truth) &
                     (d.truth <= d.est + 1.96 * d.se)).mean(),
    })

print("✓ Engine ready: declare a design, then diagnose(design) for its diagnosands.")

`diagnose()` is the whole idea in one picture: take the design, replay it many
times, and read its behavior off the pile of imagined runs.

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_10_1.png" width="55%"/></center>

*A design's four parts — Model, Inquiry, Data strategy, Answer strategy — replayed across k simulated runs; comparing those runs to the truth you built in is exactly what Monte-Carlo diagnosis does. (Figure from the replication materials of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences* (MIT-licensed archive; the book is free at book.declaredesign.org).)*

### 🔮 Pause & Predict

We are about to diagnose the textbook mentoring design — the clean, randomized
two-group study, n = 100, a true 2-point effect. **Before running**, commit
predictions in the cell below for each diagnosand. Will this design be **biased**
(estimate systematically off from 2)? What is its **power** — will it detect the
real effect most of the time, or rarely? And will its 95% intervals hit the truth
about **95%** of the time?

### YOUR ANSWER HERE:

**Bias (near 0, or systematically off?):**

**Power (roughly what % of the time does it detect the effect?):**

**Coverage (near 95%, or not?):**

---

> 💡 **Gemini Prompt:** "This runs 1,000 imagined versions of a clean, randomized two-group study (n = 100, a true 2-point effect) and prints its bias, power, and coverage: [paste the next cell]. Predict each of the three numbers before I run it, and explain how a design can be perfectly honest — unbiased, with well-calibrated 95% intervals — and still be a poor design to actually run."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Compare Gemini's three predictions against the printed diagnosand table — is bias near 0, coverage near 0.95, and power far below both?
> - [ ] If Gemini predicted high power, confirm against the printed power number and be ready to explain why 'honest' (bias, coverage) and 'sensitive' (power) are different virtues.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Diagnose the textbook design: 1,000 imagined studies, summarized in three numbers.
canonical = diagnose(lambda r: two_group_design(r, n=100, effect=2.0, noise=2.0))
print("Diagnosis of the randomized two-group design (n=100, effect=2, noise=2):")
print(canonical.round(3).to_string())

# Self-checks: a clean randomized design should be ~unbiased with ~95% coverage.
assert abs(canonical["bias"]) < 0.15, "randomized design should be ~unbiased"
assert 0.92 < canonical["coverage"] < 0.98, "coverage should be ~0.95"
print("\n✓ Self-check passed: bias ≈ 0 and coverage ≈ 0.95 — the design is HONEST.")
print(f"  But power is only {canonical['power']:.0%}: honest, and nearly blind.")

The three diagnosands map onto a picture worth memorizing — bias versus precision,
drawn as darts at a target:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_10_5.png" width="80%"/></center>

*Three ways a design's estimates can land: unbiased but imprecise (centered on the estimand, widely scattered), unbiased and precise (centered and tight), and biased but precise (tight but off-center). SD measures the scatter, Bias the distance off-center, and RMSE combines the two. (Figure from the replication materials of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences* (MIT-licensed archive; the book is free at book.declaredesign.org).)*

### 🔍 Reading the Evidence

The diagnosis is back. In the cell below, read it like a chart in a paper. First:
two of the three diagnosands look great and one looks alarming — say which is which,
and what each good number *earns* the design the right to say. Second: the power
number should feel familiar from Friday — explain, in one sentence, why a design
can be perfectly honest (unbiased, well-calibrated intervals) and still be a bad
design to run. Third: which single diagnosand would you most want to change, and
which lever from nb10 (n, noise, effect) would you reach for first?

### YOUR ANSWER HERE:

**The two good diagnosands and the one alarming one (and what the good ones earn):**

**Why an honest design can still be a bad one to run:**

**The diagnosand I'd most want to change, and the lever I'd reach for:**

---

The flight-simulator argument, made concrete: the diagnosis just crashed this
design in the computer — it revealed a design that would fail to find its own real
effect most of the time — and the only cost was a fraction of a second. Now let's
learn to recognize the three ways a design can be sick, so you can spot the disease
from the diagnosands alone.

> **A question that often comes up here:** *"Coverage came back around 95% — doesn't
> that just mean I can be 95% sure my own answer is right?"* No, and the difference
> is the heart of honest inference. Coverage is a property of the *procedure* across
> many imagined runs: build an interval this way a thousand times and about 950 of
> them contain the truth. Your one real interval either contains the truth or it does
> not, and you never learn which. Good coverage earns you the right to *report* an
> interval and trust the method that made it; it never turns your single interval
> into a 95% bet on being correct.

---

### 🛠️ Run the Study — you choose the parameters

Your turn to point the engine. In the cell below, change **only the three
parameters** — `MY_N`, `MY_EFFECT`, `MY_NOISE` — to values close to your own
project's design, then run the diagnosis. You are not touching the machinery; you
are the researcher choosing where to aim it. Try to find a combination whose power
clears 80% (a common target), and notice what you had to spend to get there.

In [ ]:
# YOUR SOLUTION HERE — set these three to match (a simplified version of) your design.
MY_N = 100        # how many people you can realistically reach
MY_EFFECT = 2.0   # the smallest effect you would care to detect
MY_NOISE = 2.0    # extra measurement noise on top of natural variation

mine = diagnose(lambda r: two_group_design(r, n=MY_N, effect=MY_EFFECT, noise=MY_NOISE))
print(f"Your design (n={MY_N}, effect={MY_EFFECT}, noise={MY_NOISE}):")
print(mine.round(3).to_string())
print(f"\nPower clears 80%? {'✓ yes' if mine['power'] >= 0.80 else '✗ not yet — keep tuning'}")

### 📝 Practice — match the sick design to its disease

The cell below diagnoses **three broken versions** of the mentoring study. Each has
exactly one disease. Read the three diagnosand tables, then in the cell that
follows, match each design (**X**, **Y**, **Z**) to its disease and name the one
number that gives it away:

- **biased** — the estimate is systematically off from the truth;
- **underpowered** — honest, but almost never detects a real effect;
- **bad coverage** — the 95% intervals miss the truth far more than 5% of the time
  (overconfident: the design *feels* sure and is often wrong).

> 💡 **Gemini Prompt:** "This code diagnoses three broken versions of the same study, each sick in exactly one way — one biased, one underpowered, one with intervals that are too narrow (bad coverage): [paste the next cell]. Without running it, describe which single diagnosand would give away each of those three diseases, and roughly what value that number would take when the disease is present."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Match Gemini's predicted giveaways against the three printed diagnosand rows — does the biased design show a large bias, the underpowered one a tiny power, the bad-coverage one a coverage far from 0.95?
> - [ ] Watch the trap it may miss: the bad-coverage design can show HIGHER power than the honest one — confirm that in the output and be ready to say why understating uncertainty fakes confidence.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Three designs, each sick in exactly one way. (Read the diseases OFF the tables.)

# Design X: the eager-volunteer problem. Instead of randomizing, the people already
# highest on belonging (top half by Y0) are the ones who sign up for a mentor.
def design_X(rng, n=100, effect=2.0, noise=2.0):
    world = make_world(n, effect, noise, rng).sort_values("Y0").reset_index(drop=True)
    z = np.zeros(n, dtype=int); z[n // 2:] = 1          # high-baseline half self-selects in
    world["Z"] = z
    world["Y"] = np.where(world["Z"] == 1, world["Y1"], world["Y0"])
    est, se = difference_in_means(world)
    return est, se, effect

# Design Y: a clean randomized study, but with only 24 people.
def design_Y(rng):
    return two_group_design(rng, n=24, effect=2.0, noise=2.0)

# Design Z: a clean, well-powered randomized study whose analyst reports a standard
# error only HALF as large as it should be (forgot the control group's variability).
def design_Z(rng):
    est, se, truth = two_group_design(rng, n=100, effect=2.0, noise=2.0)
    return est, se * 0.5, truth

for name, fn in [("X", design_X), ("Y", design_Y), ("Z", design_Z)]:
    print(f"Design {name}:", diagnose(fn).round(3).to_dict())

### YOUR ANSWER HERE:

**Design X is _______ — the giveaway number is _______.**

**Design Y is _______ — the giveaway number is _______.**

**Design Z is _______ — the giveaway number is _______.**

---

### ⚖️ Make a Design Choice

You now have three diagnosands and a feel for what each catches. In the cell below,
pick the **one diagnosand your own project most needs to protect**, and defend it in
a short paragraph. A causal claim on an observational design lives or dies on
**bias**. A small pilot chasing a subtle effect lives or dies on **power**. A study
that will report an interval to decision-makers lives or dies on **coverage**. Name
yours, say why it is the one that would sink you if it went wrong, and state the
number you would need to see to trust your design.

### YOUR ANSWER HERE:

**The diagnosand my project most needs to protect:**

**Why it is the one that would sink me, and the value I need to see:**

---

### 🎯 Project Transfer — declare your own design (M08 begins)

Time to point the engine at your own project. In the cell below, write the **plain-
language declaration** of a simplified version of your design, then translate it
into the three parameters the engine needs. You are not writing code from scratch —
you are filling `MY_N`, `MY_EFFECT`, `MY_NOISE` (you did this in the 🛠️ cell above)
and recording what real thing each stands for in your study.

### YOUR ANSWER HERE:

**My design in one plain sentence (units, comparison, outcome):**

**n stands for → my value:** &nbsp; **effect stands for → my value:** &nbsp; **noise stands for → my value:**

**The diagnosand result I got, and my first read of it:**

---

### 🎟️ Claim Ticket

**Claim Ticket #23** — the diagnosand your own design most needs checked, and one
sentence on why. Bring this into Wednesday, where we fix what the diagnosis exposes.

### YOUR ANSWER HERE:

**The diagnosand my design most needs checked, and why:**

---

---

*(Meeting M24 picks up here — Wed Oct 21.)*

## 2. Redesign: Fix It Before You Run It

**Guiding question:** *what is the cheapest change that turns a design I can't
defend into one I can — and can I prove it with the same simulation that scared me?*

> *"A diagnosis without a redesign is just anxiety. You've told me your study has a
> 15% chance of finding what it's looking for — fine. Now show me the cheapest
> change that gets you to a number you can defend, and prove it with the same
> simulation that scared you. Compare designs, not vibes."*
> — a **grant panel reviewer**, refusing to fund a worry

Monday's diagnosis found the disease: the mentoring design is honest but
underpowered. Redesign is the cure, and it has exactly four levers — each with a
price tag:

- **More units (n)** — the reliable lever. Price: recruitment, time, sometimes money.
- **Less noise** — cleaner measurement, tighter procedures. Price: better
  instruments and more careful data collection.
- **A bigger effect** — a stronger "dose" of the intervention. Price: often not
  yours to set, and a bigger dose can change what you are even studying.
- **A better estimator or assignment** — blocking, covariate adjustment, matching.
  Price: added assumptions and complexity.

The engine lets you *price* these levers instead of guessing. We will rerun the
diagnosis across a grid of designs, put the results in one comparison table, and
read the cheapest fix straight off it.

> **A question that often comes up here:** *"When is it OK to just shrink my
> question so my weak design can answer it?"* When you say so, out loud, in the
> declaration. Narrowing "does mentoring raise belonging for all first-years?" to
> "…for first-generation first-years in the honors dorm?" can be a perfectly honest
> redesign — a smaller, answerable question. It becomes dishonest only when it
> happens *silently*, after the fact, so a reader thinks you answered the big
> question when you quietly swapped in a small one. Honest redesign is a new
> declaration; silent redesign is a bait-and-switch.

---

### 🔮 Pause & Predict

We will rerun the diagnosis across a grid: sample sizes n ∈ {100, 200, 400} crossed
with two measurement-noise levels {2, 10}, holding the true effect at 2. **Before
running**, commit in the cell below: as n grows, what happens to **power** and to
**bias**? And between the two noise levels, which column will have higher power —
the noisy one or the clean one?

### YOUR ANSWER HERE:

**As n grows: power does ____, bias does ____.**

**Higher power: the noise = 2 column or the noise = 10 column? Why:**

---

> 💡 **Gemini Prompt:** "This reruns the diagnosis across a grid of sample sizes crossed with two measurement-noise levels, holding the true effect at 2: [paste the next cell]. Predict what happens to power as n grows, what happens to bias across the whole grid, and whether the low-noise column or the high-noise column will have more power — and explain why cutting noise might buy surprisingly little here."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Read the comparison table: does power rise with n at fixed noise, and does the bias column stay near 0 everywhere, as Gemini predicted?
> - [ ] Check how little the noise cut actually bought at n = 100, and be ready to explain why the outcome's large natural spread dominates the noise term.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Redesign grid: diagnose the design at every (n, noise) combination, effect fixed at 2.
grid_ns = [100, 200, 400]
grid_noises = [2, 10]

records = []
for n in grid_ns:
    for noise in grid_noises:
        d = diagnose(lambda r, n=n, noise=noise: two_group_design(r, n=n, effect=2.0, noise=noise))
        records.append({"n": n, "noise": noise, "bias": d["bias"],
                        "power": d["power"], "coverage": d["coverage"]})

comparison = pd.DataFrame(records).round(3)
print("Redesign comparison (true effect = 2 throughout):")
print(comparison.to_string(index=False))

# Self-checks: at fixed noise, power must rise with n; the randomized design stays unbiased.
p100 = comparison.query("n == 100 and noise == 2")["power"].iloc[0]
p400 = comparison.query("n == 400 and noise == 2")["power"].iloc[0]
assert p400 > p100, "power should increase with n"
assert comparison["bias"].abs().max() < 0.15, "randomized design should stay ~unbiased"
print(f"\n✓ Self-check passed: power rises with n ({p100:.0%} → {p400:.0%}), "
      f"and |bias| stays under 0.15 everywhere (max {comparison['bias'].abs().max():.3f}).")

Power against sample size is the redesign curve you are reading — and the textbook
draws it with the target most fields aim for:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_11_1.png" width="70%"/></center>

*As the data strategy's sample size grows, the diagnosand power rises toward — and eventually past — the conventional 0.8 target. Reading where your design's curve crosses that line is how you price "more people" against the power you actually need. (Figure from the replication materials of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences* (MIT-licensed archive; the book is free at book.declaredesign.org).)*

### 🔍 Reading the Evidence

Read the comparison table in the cell below. First: follow the **power** column down
the noise = 2 rows — how much does each doubling of n buy you, and where would you
have to land to clear 80%? Second: compare the two noise levels at n = 100 — cutting
noise from 10 to 2 raises power by how much, and why is that gain so *small* next to
the gain from more people? (Recall the outcome's natural spread is already large.)
Third: the **bias** column barely moves across the whole grid — what does that tell
you about which problems redesign-by-parameter can and cannot fix?

### YOUR ANSWER HERE:

**What each doubling of n buys, and where 80% power lives:**

**How much cutting noise (10 → 2) helped at n = 100, and why it's small:**

**What the flat bias column says redesign-by-parameter cannot fix:**

---

### 📝 Practice — spend the budget, then verify

Your mentoring design sits at n = 100, noise = 10, power \~0.12. A grant funds
**one** upgrade, and the two candidates cost the same:

- **Fix 1 — double the sample:** n = 100 → 200, noise stays 10.
- **Fix 2 — clean up measurement:** noise = 10 → 2, n stays 100.

Predict which buys more power, then read the two numbers off the comparison table
above (or run the verification cell) and confirm.

> 💡 **Gemini Prompt:** "I have two equal-cost fixes for an underpowered study: double the sample (n 100 → 200, noise stays 10), or clean up measurement (noise 10 → 2, n stays 100). Predict which one buys more statistical power, and explain your reasoning about where this study's uncertainty is mostly coming from."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Compare Gemini's prediction against the two printed power numbers (Fix 1 vs Fix 2) — which fix actually won, and by how much?
> - [ ] If its reasoning assumed cleaning measurement is the obvious win, check that against the result: price both fixes with the SAME simulation rather than trusting the intuition.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# YOUR SOLUTION HERE — compare the two equal-cost fixes head to head.
base  = diagnose(lambda r: two_group_design(r, n=100, effect=2.0, noise=10))["power"]
fix_1 = diagnose(lambda r: two_group_design(r, n=200, effect=2.0, noise=10))["power"]  # double n
fix_2 = diagnose(lambda r: two_group_design(r, n=100, effect=2.0, noise=2))["power"]   # clean measure
print(f"Baseline (n=100, noise=10): power = {base:.0%}")
print(f"Fix 1  double n     -> {fix_1:.0%}")
print(f"Fix 2  clean measure -> {fix_2:.0%}")
print(f"\nCheapest winning fix: {'double n' if fix_1 > fix_2 else 'clean measurement'}")

### YOUR ANSWER HERE:

**My prediction (which fix buys more power):**

**What the numbers showed:**

---

### 🎯 Project Transfer — your redesign, with before/after numbers (M08)

This is the heart of your M08 memo. In the cell below, record your design's
diagnosis (the before), the **one** redesign lever you will pull, and the diagnosis
after pulling it (the after). Use the 🛠️ engine cell from Monday for both runs —
change one parameter, rerun, and copy the two diagnosand rows here. Then write the
one-sentence justification a grant reviewer would accept: *"Redesign R raises
[diagnosand] from ___ to ___ at cost ___."*

> 💡 **Gemini Prompt:** "Here is my design's diagnosis:
> [paste bias, power, coverage]. Here is the redesign I'm considering: [describe].
> Argue the case *against* my redesign — what cheaper fix might I be overlooking,
> and what could my before/after comparison be hiding?"
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check any cheaper fix it proposes by actually running it through
>   `diagnose()` — do not take the AI's word that it helps.
> - [ ] Confirm your before/after numbers came from YOUR simulation, not a plausible
>   number the AI supplied.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**Before (my design's diagnosis — bias / power / coverage):**

**The one lever I will pull (and its price):**

**After (diagnosis once I pull it):**

**My one-sentence justification ("Redesign R raises ___ from ___ to ___ at cost ___"):**

---

### 🎟️ Claim Ticket

**Claim Ticket #24** — your design's diagnosed weakness and the fix you committed
to, **with numbers** (before → after). This is the spine of the M08 memo you submit
Friday.

### YOUR ANSWER HERE:

**My diagnosed weakness + committed fix, with before → after numbers:**

---

## 3. Wrap-Up

You built and drove the course's diagnosis engine. You learned to **declare** a
design as a function that plays out one study, to **diagnose** it across 1,000
imagined runs, and to read the three diagnosands — **bias** (does it aim true?),
**power** (does it see?), **coverage** (does its interval keep its promise?). You
matched three sick designs to their diseases and saw that bias is *structural* while
power and coverage bend to *parameters*. Then you **redesigned**: priced the four
levers with the same simulation that exposed the weakness, and defended the cheapest
fix with before/after numbers instead of vibes — while keeping the line between an
honest redesign and a silent bait-and-switch bright.

> **"An untested design has undiagnosed diseases. Crash it in the simulator, read
> the diagnosands, and fix the cheapest thing that matters — before you spend a
> single person's time."**

On Friday, nb12 opens the **prediction** deep dive — a different column of the
four-approach map, where the question is not "what is the effect?" but "can I
forecast new cases better than the dumbest honest rule?" You will carry the same
habit there: never trust a number you have not stress-tested. This notebook served
milestone M08 — your Design Diagnosis & Redesign Memo, due Friday.

---

## 4. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 10 'Diagnosing designs' + diagnosis_10.1 | Monte-Carlo diagnosis | the declare/diagnose loop translated to a compact inline Python helper | translated (R→Python)*
- *RDSS ch. 11 'Redesigning' + declaration_11.1 | redesign across a parameter grid | redesign-comparison loop translated to a grid rerun | translated (R→Python)*
- *RDSS ch. 13 'Designing in code' | design as code | the declare-a-design-as-a-function pattern | adapted (concept-level, honors non-quant audience)*
- *nb04 mentoring-program world (make_world) | course-internal | potential-outcomes world reused as the diagnosis sandbox | reused*

**Dataset attribution:** the simulations in this notebook use no external dataset;
the mentoring world is generated in code. Where course datasets are used elsewhere,
they come from the `rdss` package (Blair, Coppock & Humphreys, MIT License),
companion to *Research Design in the Social Sciences* (2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 10, 11, and 13. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>